# 74장 · 같은 서비스를 EC2 배포로 이어가기

로컬에서 열리던 과정 안내 API를 EC2에 올린다고 가정합니다. 이미지뿐 아니라 어느 프로세스가 외부 요청을 받고, 어느 컴퓨터의 저장소를 읽는지 정해야 합니다. 수업용 배포 설정을 읽고, 실제 접속을 확인하려면 어떤 기록이 필요한지 짚어 봅니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 외부 요청이 도착할 목적지 찾기

배포 예제에서는 HTTPS를 받는 프록시만 호스트의 443번에 연결합니다. 프록시는 Compose 네트워크 안의 api:8000으로 전달하고, API는 내부 db:5432를 사용합니다. DB 접속 정보를 브라우저에 건네지 않습니다.



**볼 곳**　‘외부 HTTPS’에서 출발해 ‘프록시 443’를 거친 뒤 ‘서비스’가 되는 순서를 따라가세요.

**애니메이션의 의미**　외부 HTTPS 요청은 프록시를 거쳐 서비스에 도달합니다.


### 셀 01 · 배포 선언에서 공개 포트 확인

기존 배포 설정에서 포트 관련 부분만 가져왔습니다. 어떤 서비스가 외부 요청의 첫 번째 목적지인가요?

EC2는 클라우드에서 빌려 쓰는 가상 컴퓨터입니다. 학생 브라우저에서 접속하는 주소와 그 컴퓨터 안에서 프록시가 API로 연결하는 주소는 서로 다른 접속점입니다. 연결표에서 브라우저가 출발점인 줄을 먼저 보세요.

In [1]:
services = {"proxy": {"ports": ["443:443"]}, "api": {}, "db": {}}
published = {name: spec["ports"] for name, spec in services.items() if "ports" in spec}
print(published)

{'proxy': ['443:443']}


proxy만 남습니다. EC2 보안 그룹, 도메인, 인증서, 프록시가 모두 준비돼야 실제 외부 HTTPS 요청까지 도착합니다. 로컬 선언만으로 그 상태를 확인할 수 없습니다.

### 과제 02 · 프록시가 사용할 API 주소 읽기

설정에서 실제 전달할 내부 주소를 추출하세요. 그 값이 학생 브라우저에 넣을 주소와 같은지 설명해 보세요.

In [2]:
directive = 'proxy_pass http://api:8000;'
# 지시문의 주소에서 끝의 세미콜론을 떼세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>directive = &#x27;proxy_pass http://api:8000;&#x27;
print(directive.split()[1].rstrip(&#x27;;&#x27;))</code></pre>
<p>실행 결과</p><pre><code>http://api:8000</code></pre>
<p>http://api:8000은 Compose 네트워크 안에서 쓰는 이름입니다. 브라우저는 강사가 제공한 HTTPS 도메인을 사용합니다.</p>
</details>

## 로컬 이미지와 EC2 배포 결과 구분하기

이미지 태그가 같아도 내용이 바뀔 수 있습니다. 배포 준비 단계에서는 검토한 이미지 digest, 호스트 플랫폼, 데이터 이관 여부를 기록합니다. 실제 수업 계정에서는 EC2 인스턴스·볼륨의 소유자와 비용 정리 책임도 같이 적습니다.



**볼 곳**　‘배포 근거’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘로컬 이미지’, 아래쪽은 ‘EC2 응답’입니다.

**애니메이션의 의미**　로컬 이미지가 있다는 것과 배포 응답이 온 것은 별개입니다.


### 셀 03 · 선택 실습 · 배포된 서비스의 health 읽기

수업에 배정된 HTTPS 주소가 있을 때만 base_url에 입력하세요. 로컬 실습 기본값으로는 요청을 보내지 않습니다. 서버 본문 전체 대신 HTTP 코드와 health의 application만 확인합니다.

In [3]:
import json
from urllib.parse import urlsplit
from urllib.request import urlopen
base_url = ""
if base_url:
    if urlsplit(base_url).scheme != "https":
        raise ValueError("수업 서비스의 HTTPS 주소를 넣으세요.")
    with urlopen(base_url.rstrip("/") + "/health", timeout=5) as reply:
        body = json.load(reply)
        print(reply.status, body.get("application"))
else:
    print("배정된 배포 주소가 없어 HTTPS 확인을 건너뜁니다.")

배정된 배포 주소가 없어 HTTPS 확인을 건너뜁니다.


기본값의 안내는 미실행을 뜻합니다. 실제 요청에서 200이 와도 앞서 저장한 문서가 옮겨졌는지, 답변이 같은 근거를 인용하는지는 해당 요청으로 별도 확인해야 합니다. 수업에서 배정한 HTTPS 주소를 base_url에 넣고 /health 응답의 상태 코드와 application 값을 함께 적으세요. 주소가 없는 상태에서 설정을 읽은 경우 확인한 값은 포트와 프록시 대상까지입니다.

## 돌아보기

실제 운영 기록에는 접속 시각·대상·응답 코드와 함께 어느 호스트의 어느 볼륨을 사용했는지 남기세요. 새 EC2에서 같은 이름의 빈 볼륨을 만든 것은 원본 데이터 이관 증거가 아닙니다.